In [1]:
import pandas as pd
import numpy as np

def custom_minmax_scaler(series, feature_name, lower_bound_q=0.01, upper_bound_q=0.95, min_val=0.1, max_val=1.0):
    """
    自定義正規化函數：包含 1%~95% 縮尾處理，並映射至 [0.1, 1.0] 區間。
    """
    # 縮尾處理 (Winsorization) 消除極端離群值
    lower_bound = series.quantile(lower_bound_q)
    upper_bound = series.quantile(upper_bound_q)
    clipped = series.clip(lower=lower_bound, upper=upper_bound)
    
    # 壓縮至 0~1
    s_min = clipped.min()
    s_max = clipped.max()
    
    if s_max == s_min:
        scaled = np.zeros(len(clipped))
    else:
        scaled = (clipped - s_min) / (s_max - s_min)
        
    # 線性映射至 [0.1, 1.0]
    final_scaled = min_val + scaled * (max_val - min_val)
    return final_scaled

def run_stage0_normalization_and_reduction():
    print("啟動 Stage 0: DEA 前置正規化與客觀降維...")
    
    try:
        df = pd.read_csv("csv\\stage0_final_matrix.csv")
    except FileNotFoundError:
        print("❌ 找不到 stage0_final_matrix.csv。")
        return
        
    df_dea = pd.DataFrame({'ETF': df['ETF']})
    
    # ==========================================
    # 1. DEA 產出項 (Outputs): 越大越好
    # ==========================================
    
    # [報酬維度 R_P]
    norm_cagr = custom_minmax_scaler(df['Return_CAGR (%)'], 'CAGR')
    norm_div = custom_minmax_scaler(df['Return_Div (%)'], 'Div')
    df_dea['Out_Return'] = (norm_cagr + norm_div) / 2
    
    # [流動性維度 L_P] - 先取對數處理嚴重右偏
    log_volume = np.log1p(df['Liq_Volume (M)'])
    log_aum = np.log1p(df['Liq_AUM (B)'])
    norm_vol = custom_minmax_scaler(log_volume, 'Volume')
    norm_aum = custom_minmax_scaler(log_aum, 'AUM')
    df_dea['Out_Liquidity'] = (norm_vol + norm_aum) / 2
    
    # [分散度維度 D_P]
    df_dea['Out_Diversity'] = custom_minmax_scaler(df['Div_Score (產出)'], 'Diversity')
    
    # [市場情緒維度 S_P]
    df_dea['Out_Sentiment'] = custom_minmax_scaler(df['FinBERT_score'], 'Sentiment', lower_bound_q=0.05, upper_bound_q=0.95)
    
    # ==========================================
    # 2. DEA 投入項 (Inputs): 越小越好 (消耗的資源/承擔的風險)
    # ==========================================
    
    # [風險維度 V_P] - MaxDD 取絕對值代表回撤幅度
    norm_risk_vol = custom_minmax_scaler(df['Risk_Vol (%)'], 'Risk_Vol')
    norm_maxdd = custom_minmax_scaler(df['Risk_MaxDD (%)'].abs(), 'Risk_MaxDD')
    df_dea['In_Risk'] = (norm_risk_vol + norm_maxdd) / 2
    
    # [成本維度 C_P]
    df_dea['In_Cost'] = custom_minmax_scaler(df['Cost_ExpRatio (%)'], 'Cost')
    
    # ==========================================
    # 3. 儲存 DEA 專用矩陣
    # ==========================================
    
    # 重新排列欄位，便於後續切片
    cols = ['ETF', 'In_Risk', 'In_Cost', 'Out_Return', 'Out_Liquidity', 'Out_Diversity', 'Out_Sentiment']
    df_dea = df_dea[cols]
    
    print("\n=== 📊 降維後 DEA 矩陣預覽 (皆落於 0.1 ~ 1.0 區間) ===")
    print(df_dea.head().round(4).to_string())
    
    df_dea.to_csv("csv\\stage0_dea_ready_matrix.csv", index=False)
    print("\n✅ 已輸出 stage0_dea_ready_matrix.csv")

# 執行
run_stage0_normalization_and_reduction()

啟動 Stage 0: DEA 前置正規化與客觀降維...

=== 📊 降維後 DEA 矩陣預覽 (皆落於 0.1 ~ 1.0 區間) ===
   ETF  In_Risk  In_Cost  Out_Return  Out_Liquidity  Out_Diversity  Out_Sentiment
0  VOO   0.3079   0.1000      0.4717         0.9046         0.6598         0.3868
1  IVV   0.3103   0.1000      0.4760         0.8976         0.6437         0.2904
2  SPY   0.3157   0.1813      0.4664         1.0000         0.6445         0.3729
3  VTI   0.3285   0.1000      0.4656         0.8205         0.6764         0.4052
4  QQQ   0.4590   0.2875      0.4824         1.0000         0.2961         0.3756

✅ 已輸出 stage0_dea_ready_matrix.csv


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def run_post_normalization_eda():
    print("啟動正規化數據視覺化檢驗...")
    try:
        df = pd.read_csv("csv\\stage0_dea_ready_matrix.csv")
    except FileNotFoundError:
        print("❌ 找不到 stage0_dea_ready_matrix.csv，請確認前一階段已執行。")
        return

    features = ['In_Risk', 'In_Cost', 'Out_Return', 'Out_Liquidity', 'Out_Diversity', 'Out_Sentiment']
    
    # 設置繪圖風格與中文字型
    sns.set_theme(style="whitegrid")
    plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'PingFang HK', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False

    # ==========================================
    # 1. 繪製直方圖 (檢驗分佈形狀)
    # ==========================================
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
    axes = axes.flatten()

    for i, feature in enumerate(features):
        sns.histplot(df[feature], kde=True, ax=axes[i], color='teal', bins=20, alpha=0.6)
        axes[i].set_title(f'正規化分佈: {feature}', fontweight='bold')
        axes[i].set_xlim(0, 1.1)  # 強制鎖定 X 軸範圍以便檢查邊界
        axes[i].set_xlabel('數值區間 [0.1, 1.0]')
        axes[i].set_ylabel('頻率')

    plt.tight_layout()
    plt.savefig("png\\eda_normalized_histograms.png", dpi=300)
    plt.close()
    print("✅ 產出正規化特徵分佈圖：eda_normalized_histograms.png")
    
    # ==========================================
    # 2. 繪製箱型圖 (檢驗極端值是否消除)
    # ==========================================
    fig_box, axes_box = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
    axes_box = axes_box.flatten()
    
    for i, feature in enumerate(features):
        sns.boxplot(x=df[feature], ax=axes_box[i], color='mediumaquamarine')
        axes_box[i].set_title(f'邊界檢驗: {feature}', fontweight='bold')
        axes_box[i].set_xlim(0, 1.1)
        
    plt.tight_layout()
    plt.savefig("png\\eda_normalized_boxplots.png", dpi=300)
    plt.close()
    print("✅ 產出正規化特徵箱型圖：eda_normalized_boxplots.png")

# 執行
run_post_normalization_eda()

啟動正規化數據視覺化檢驗...
✅ 產出正規化特徵分佈圖：eda_normalized_histograms.png
✅ 產出正規化特徵箱型圖：eda_normalized_boxplots.png
